In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.api as sm

# Create output folders inside your working directory
os.makedirs("data", exist_ok=True)
os.makedirs("visuals", exist_ok=True)

# 1. LOAD & CLEAN DATA (UCI Bank Marketing uses semicolon delimiter)
df = pd.read_csv("bank-full.csv", sep=";")

# Standardize binary target & engineer funnel stages
df["Converted"] = (df["y"] == "yes").astype(int)
df["Engaged_Lead"] = (df["duration"] >= 180).astype(int)       # >= 3 mins meaningful call engagement
df["High_Intent_Lead"] = (df["duration"] >= 300).astype(int)   # >= 5 mins deep product discussion
df["Repeat_Outreach"] = (df["previous"] > 0).astype(int)

# Categorize campaign touchpoint frequency
df["Touchpoint_Bucket"] = pd.cut(
    df["campaign"],
    bins=[0, 1, 2, 3, 5, 100],
    labels=["1 Touch", "2 Touches", "3 Touches", "4–5 Touches", "6+ Touches"]
)

# Save cleaned dataset
df.to_csv("data/marketing_funnel_cleaned.csv", index=False)

# 2. MULTI-STAGE MARKETING FUNNEL METRICS
total_leads = len(df)
known_channel = (df["contact"] != "unknown").sum()
engaged_leads = df["Engaged_Lead"].sum()
high_intent_leads = df["High_Intent_Lead"].sum()
converted_customers = df["Converted"].sum()

funnel_stages = pd.DataFrame({
    "Stage": [
        "1. Targeted Campaign Leads",
        "2. Verified Contact Channel",
        "3. Engaged Leads (>=3 Min Call)",
        "4. High-Intent Leads (>=5 Min Call)",
        "5. Converted Customers (Subscribed)"
    ],
    "Count": [total_leads, known_channel, engaged_leads, high_intent_leads, converted_customers]
})
funnel_stages["Pct_of_Total (%)"] = (funnel_stages["Count"] / total_leads) * 100
funnel_stages["Stage_Conversion (%)"] = (
    funnel_stages["Count"] / funnel_stages["Count"].shift(1).fillna(total_leads)
) * 100
funnel_stages["Stage_Dropoff (%)"] = 100 - funnel_stages["Stage_Conversion (%)"]

print("=" * 68)
print("EXECUTIVE MARKETING FUNNEL & CONVERSION SUMMARY")
print("=" * 68)
print(f"Total Targeted Leads:          {total_leads:,}")
print(f"Total Converted Customers:     {converted_customers:,}")
print(f"Overall Funnel Conversion:     {(converted_customers / total_leads)*100:.2f}%")
print(f"Avg Engagement (Converted):    {df.loc[df['Converted']==1, 'duration'].mean():.1f} sec")
print(f"Avg Engagement (Non-Converted):{df.loc[df['Converted']==0, 'duration'].mean():.1f} sec")
print("=" * 68)
print("\n--- STAGE-BY-STAGE FUNNEL PROGRESSION ---")
print(funnel_stages.round(2).to_string(index=False))

print("\n--- CONVERSION BY ACQUISITION CHANNEL ---")
channel_summary = df.groupby("contact").agg(
    Leads=("Converted", "count"),
    Conversions=("Converted", "sum"),
    Conversion_Rate_Pct=("Converted", lambda x: x.mean() * 100),
    Avg_Duration_Sec=("duration", "mean")
).sort_values(by="Conversion_Rate_Pct", ascending=False)
print(channel_summary.round(2))

print("\n--- CONVERSION BY PRIOR CAMPAIGN OUTCOME (RETENTION/RETARGETING FUNNEL) ---")
poutcome_summary = df.groupby("poutcome").agg(
    Leads=("Converted", "count"),
    Conversions=("Converted", "sum"),
    Conversion_Rate_Pct=("Converted", lambda x: x.mean() * 100)
).sort_values(by="Conversion_Rate_Pct", ascending=False)
print(poutcome_summary.round(2))

print("\n--- CONVERSION BY CAMPAIGN TOUCHPOINT FREQUENCY ---")
touch_summary = df.groupby("Touchpoint_Bucket", observed=False).agg(
    Leads=("Converted", "count"),
    Conversions=("Converted", "sum"),
    Conversion_Rate_Pct=("Converted", lambda x: x.mean() * 100)
)
print(touch_summary.round(2))

print("\n--- TOP 5 & BOTTOM 5 MONTHS BY CONVERSION RATE ---")
month_order = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
month_summary = df.groupby("month").agg(
    Leads=("Converted", "count"),
    Conversions=("Converted", "sum"),
    Conversion_Rate_Pct=("Converted", lambda x: x.mean() * 100)
).reindex(month_order)
print(month_summary.round(2))

# 3. STATISTICAL ANALYSIS: LOGISTIC REGRESSION (CONVERSION DRIVERS)
model_df = pd.DataFrame({
    "duration_mins": df["duration"] / 60.0,
    "campaign_touches": df["campaign"],
    "Channel_Cellular": (df["contact"] == "cellular").astype(int),
    "Channel_Telephone": (df["contact"] == "telephone").astype(int),
    "Prior_Success": (df["poutcome"] == "success").astype(int),
    "Housing_Loan": (df["housing"] == "yes").astype(int),
    "Personal_Loan": (df["loan"] == "yes").astype(int)
})
X = sm.add_constant(model_df)
y = df["Converted"]
logit_model = sm.Logit(y, X).fit(disp=False)

odds_ratios = pd.DataFrame({
    "Coef": logit_model.params,
    "Odds_Ratio": np.exp(logit_model.params),
    "P_Value": logit_model.pvalues
})
print("\n--- LOGISTIC REGRESSION: FUNNEL CONVERSION DRIVERS (ODDS RATIOS) ---")
print(odds_ratios.round(4))

# 4. GENERATE CLIENT-READY VISUALIZATIONS
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# CHART 1: Marketing Funnel Drop-Off Visualization
fig, ax = plt.subplots(figsize=(10.5, 5.2))
stages_rev = funnel_stages.iloc[::-1]
colors_f = ["#2e8b57", "#3b7a9e", "#2b5c8f", "#1f4e78", "#163859"]
bars = ax.barh(stages_rev["Stage"], stages_rev["Count"], color=colors_f, height=0.58)
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.set_title("Marketing Funnel Progression & Stage Conversion Rates", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Number of Leads / Customers")
ax.grid(axis="x", linestyle="--", alpha=0.4)
for bar, pct in zip(bars, stages_rev["Pct_of_Total (%)"]):
    w = bar.get_width()
    ax.text(w + 650, bar.get_y() + bar.get_height()/2, f"{w:,.0f} ({pct:.1f}%)", va="center", fontweight="bold", fontsize=9.5)
ax.set_xlim(0, total_leads * 1.22)
plt.tight_layout()
plt.savefig("visuals/01_marketing_funnel_progression.png", dpi=300)
plt.close()

# CHART 2: Conversion Rate by Channel & Prior Campaign Outcome
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))

ch_plot = channel_summary.reset_index()
b1 = axes[0].bar(ch_plot["contact"].str.title(), ch_plot["Conversion_Rate_Pct"], color=["#2e8b57", "#4a90d9", "#95a5a6"], width=0.5)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].set_title("Conversion Rate (%) by Contact Channel", fontsize=12, fontweight="bold")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)
for bar in b1:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.5, f"{h:.2f}%", ha="center", fontweight="bold")
axes[0].set_ylim(0, 18)

po_plot = poutcome_summary.reset_index()
b2 = axes[1].bar(po_plot["poutcome"].str.title(), po_plot["Conversion_Rate_Pct"], color=["#2e8b57", "#4a90d9", "#e68a00", "#95a5a6"], width=0.55)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title("Conversion Rate (%) by Prior Funnel Outcome", fontsize=12, fontweight="bold")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
for bar in b2:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 1.5, f"{h:.1f}%", ha="center", fontweight="bold")
axes[1].set_ylim(0, 74)

plt.tight_layout()
plt.savefig("visuals/02_channel_and_retargeting_conversion.png", dpi=300)
plt.close()

# CHART 3: Diminishing Returns of Repeated Outreach Touchpoints
t_plot = touch_summary.reset_index()
fig, ax = plt.subplots(figsize=(9.5, 4.8))
bars = ax.bar(t_plot["Touchpoint_Bucket"], t_plot["Conversion_Rate_Pct"], color="#2b5c8f", width=0.52)
ax.plot(t_plot["Touchpoint_Bucket"], t_plot["Conversion_Rate_Pct"], color="#c0392b", marker="o", linewidth=2)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Conversion Rate (%) by Number of Campaign Touchpoints", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Conversion Rate (%)")
ax.grid(axis="y", linestyle="--", alpha=0.4)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.4, f"{h:.2f}%", ha="center", fontweight="bold")
ax.set_ylim(0, 17.5)
plt.tight_layout()
plt.savefig("visuals/03_touchpoint_diminishing_returns.png", dpi=300)
plt.close()

# CHART 4: Monthly Campaign Volume vs. Conversion Efficiency
m_plot = month_summary.reset_index()
m_plot["Month_Label"] = m_plot["month"].str.title()
fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(m_plot["Month_Label"], m_plot["Leads"], color="#1f4e78", alpha=0.35, label="Total Leads Contacted")
ax1.set_ylabel("Total Leads Contacted", color="#1f4e78", fontweight="bold")
ax1.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))

ax2 = ax1.twinx()
ax2.plot(m_plot["Month_Label"], m_plot["Conversion_Rate_Pct"], color="#c0392b", marker="o", linewidth=2.3, label="Conversion Rate (%)")
ax2.set_ylabel("Conversion Rate (%)", color="#c0392b", fontweight="bold")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.spines["top"].set_visible(False)

ax1.set_title("Monthly Lead Volume vs. Conversion Rate Efficiency", fontsize=13, fontweight="bold", pad=12)
ax1.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig("visuals/04_monthly_volume_vs_conversion.png", dpi=300)
plt.close()

print("\nAll 4 Task 3 charts saved to visuals/ and cleaned CSV saved to data/!")

EXECUTIVE MARKETING FUNNEL & CONVERSION SUMMARY
Total Targeted Leads:          45,211
Total Converted Customers:     5,289
Overall Funnel Conversion:     11.70%
Avg Engagement (Converted):    537.3 sec
Avg Engagement (Non-Converted):221.2 sec

--- STAGE-BY-STAGE FUNNEL PROGRESSION ---
                              Stage  Count  Pct_of_Total (%)  Stage_Conversion (%)  Stage_Dropoff (%)
         1. Targeted Campaign Leads  45211            100.00                100.00               0.00
        2. Verified Contact Channel  32191             71.20                 71.20              28.80
    3. Engaged Leads (>=3 Min Call)  22674             50.15                 70.44              29.56
4. High-Intent Leads (>=5 Min Call)  12329             27.27                 54.38              45.62
5. Converted Customers (Subscribed)   5289             11.70                 42.90              57.10

--- CONVERSION BY ACQUISITION CHANNEL ---
           Leads  Conversions  Conversion_Rate_Pct  Avg_Dur